# Machine Learning Zoomcamp 2026 - Homework 3: Machine Learning for Classification

- **Dataset:** `course_lead_scoring_2026.csv`
- **Target:** `converted`


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()


## Data Preparation

- Replace missing categorical values with `'NA'`
- Replace missing numerical values with `0.0`


In [ ]:
target = 'converted'
features = df.drop(columns=[target]).copy()
categorical = features.select_dtypes(include=['object']).columns.tolist()
numerical = features.select_dtypes(exclude=['object']).columns.tolist()

features[categorical] = features[categorical].fillna('NA')
features[numerical] = features[numerical].fillna(0.0)
prepared_df = features.assign(**{target: df[target]})
prepared_df.isnull().sum()


## Question 1: Mode for industry

What is the most frequent observation (mode) for the column `industry`?


In [ ]:
mode_industry = df['industry'].mode()[0]
print('Q1 Answer (Mode for industry):', mode_industry)


## Question 2: Correlation Matrix

Compute the correlation matrix for numerical features and find the pair with biggest correlation.


In [ ]:
corr = features[numerical].corr().abs()
print(corr)

candidate_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for p1, p2 in candidate_pairs:
    print(f'Corr({p1}, {p2}) = {corr.loc[p1, p2]:.4f}')
best_pair = max(candidate_pairs, key=lambda p: corr.loc[p[0], p[1]])
print('Q2 Answer (Best pair):', best_pair)


## Split the Data

- 60% train, 20% validation, 20% test with `random_state=42`


In [ ]:
df_full_train, df_test = train_test_split(prepared_df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.pop(target).values
y_val = df_val.pop(target).values
y_test = df_test.pop(target).values


## Question 3: Mutual Information

Calculate mutual information between `converted` and categorical features on the training set.


In [ ]:
mi = {col: round(mutual_info_score(y_train, df_train[col]), 2) for col in categorical}
print('MI scores:', mi)
print('Q3 Answer (Highest MI):', max(mi, key=mi.get))


## Question 4: Logistic Regression Accuracy

Train LogisticRegression with DictVectorizer and compute accuracy on validation set.


In [ ]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
val_preds = model.predict(X_val)
orig_acc = accuracy_score(y_val, val_preds)
print('Validation accuracy:', orig_acc)
print('Q4 Answer (Rounded accuracy):', round(orig_acc, 2))


## Question 5: Feature Elimination

Exclude features one by one and calculate accuracy difference.


In [ ]:
diffs = {}
for col in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    tr_sub = df_train.drop(columns=[col])
    va_sub = df_val.drop(columns=[col])
    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(tr_sub.to_dict(orient='records'))
    X_va_sub = dv_sub.transform(va_sub.to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr_sub, y_train)
    acc = accuracy_score(y_val, m.predict(X_va_sub))
    diffs[col] = abs(orig_acc - acc)
    print(f'Without {col}: accuracy={acc:.4f}, diff={orig_acc - acc:+.4f}')

print('Q5 Answer (Smallest difference):', min(diffs, key=diffs.get))


## Question 6: Regularization Parameter C

Find the parameter C with best validation accuracy.


In [ ]:
c_vals = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}
for c in c_vals:
    m = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    c_scores[c] = round(accuracy_score(y_val, m.predict(X_val)), 3)
    print(f'C={c}: validation accuracy={c_scores[c]}')

print('Q6 Answer (Best C):', max(c_vals, key=lambda c: (c_scores[c], -c)))
